# Build VisDrone TAR Archive (Colab)

This notebook creates `visdrone_yolo.tar` from:
`MyDrive/datasets/visdrone_yolo`

Expected source structure:
- `images/train`, `images/val`
- `labels/train`, `labels/val`


In [ ]:
# Step 0: parameters
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_SRC_ROOT = DRIVE_ROOT / 'datasets' / 'visdrone_yolo'
LOCAL_SRC_ROOT = Path('/content/datasets/visdrone_yolo')

# If True, build TAR directly from Drive source (single pass over Drive files).
# This is usually faster than rsync-to-local + tar.
BUILD_FROM_DRIVE_DIRECTLY = True

LOCAL_TAR_PATH = Path('/content/visdrone_yolo.tar')
DRIVE_TAR_PATH = DRIVE_ROOT / 'datasets' / 'visdrone_yolo.tar'

# Used only when BUILD_FROM_DRIVE_DIRECTLY=False.
SYNC_FROM_DRIVE_IF_NEEDED = True
FORCE_SYNC_FROM_DRIVE = False

# TAR flags
FORCE_REBUILD_TAR = False
COPY_TAR_TO_DRIVE = True

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

print('DRIVE_SRC_ROOT =', DRIVE_SRC_ROOT)
print('LOCAL_SRC_ROOT =', LOCAL_SRC_ROOT)
print('BUILD_FROM_DRIVE_DIRECTLY =', BUILD_FROM_DRIVE_DIRECTLY)
print('LOCAL_TAR_PATH =', LOCAL_TAR_PATH)
print('DRIVE_TAR_PATH =', DRIVE_TAR_PATH)


In [ ]:
# Step 1: mount Google Drive
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print('[OK] Drive mounted')


In [ ]:
# Step 2: validate source structure and resolve effective source root
from pathlib import Path
import subprocess
import shutil
import threading
import time


def count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def count_labels(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*.txt') if p.is_file())


def is_ready_yolo(root: Path) -> bool:
    required = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in required):
        return False
    return count_images(root / 'images' / 'train') > 0 and count_images(root / 'images' / 'val') > 0


def print_split_stats(root: Path, title: str) -> None:
    print(f'[{title}] root =', root)
    print(' train images:', count_images(root / 'images' / 'train'))
    print(' val images  :', count_images(root / 'images' / 'val'))
    print(' train labels:', count_labels(root / 'labels' / 'train'))
    print(' val labels  :', count_labels(root / 'labels' / 'val'))


def run_cmd_live(cmd: list[str], heartbeat_sec: int = 20) -> None:
    print('[RUN]', ' '.join(cmd))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    done = False

    def heartbeat() -> None:
        while not done and proc.poll() is None:
            print(f'[INFO] still running... {time.strftime("%H:%M:%S")}')
            time.sleep(heartbeat_sec)

    hb = threading.Thread(target=heartbeat, daemon=True)
    hb.start()

    assert proc.stdout is not None
    for line in proc.stdout:
        line = line.rstrip('\n')
        if line:
            print(line)

    code = proc.wait()
    done = True
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")


if BUILD_FROM_DRIVE_DIRECTLY:
    if not is_ready_yolo(DRIVE_SRC_ROOT):
        raise FileNotFoundError(
            f'Drive source is not ready: {DRIVE_SRC_ROOT}. '
            'Expected images/train,val and labels/train,val.'
        )
    EFFECTIVE_SRC_ROOT = DRIVE_SRC_ROOT
    print('[OK] Using Drive source directly (no rsync step).')
else:
    if not is_ready_yolo(LOCAL_SRC_ROOT):
        if SYNC_FROM_DRIVE_IF_NEEDED:
            if not is_ready_yolo(DRIVE_SRC_ROOT):
                raise FileNotFoundError(
                    f'Drive source is not ready: {DRIVE_SRC_ROOT}. '
                    'Expected images/train,val and labels/train,val.'
                )

            if FORCE_SYNC_FROM_DRIVE and LOCAL_SRC_ROOT.exists():
                shutil.rmtree(LOCAL_SRC_ROOT, ignore_errors=True)

            if not is_ready_yolo(LOCAL_SRC_ROOT):
                LOCAL_SRC_ROOT.parent.mkdir(parents=True, exist_ok=True)
                print(f'[INFO] Sync source from Drive to local SSD: {DRIVE_SRC_ROOT} -> {LOCAL_SRC_ROOT}')
                run_cmd_live([
                    'rsync',
                    '-a',
                    '--delete',
                    '--human-readable',
                    '--info=progress2,stats2',
                    f'{DRIVE_SRC_ROOT.as_posix()}/',
                    f'{LOCAL_SRC_ROOT.as_posix()}/',
                ])
        else:
            raise FileNotFoundError(
                f'Local source is not ready: {LOCAL_SRC_ROOT}. '
                'Enable SYNC_FROM_DRIVE_IF_NEEDED=True or prepare local source manually.'
            )

    if not is_ready_yolo(LOCAL_SRC_ROOT):
        raise RuntimeError(f'Local source is still not ready: {LOCAL_SRC_ROOT}')
    EFFECTIVE_SRC_ROOT = LOCAL_SRC_ROOT

print('[OK] Effective source is ready for TAR build.')
print_split_stats(EFFECTIVE_SRC_ROOT, 'EFFECTIVE SOURCE')

In [ ]:
# Step 3: build TAR locally in /content with visible progress, then copy to Drive
import time
import tarfile
import shutil
from pathlib import Path

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

if LOCAL_TAR_PATH.exists() and FORCE_REBUILD_TAR:
    LOCAL_TAR_PATH.unlink()

if not LOCAL_TAR_PATH.exists():
    LOCAL_TAR_PATH.parent.mkdir(parents=True, exist_ok=True)

    entries = [EFFECTIVE_SRC_ROOT] + sorted(EFFECTIVE_SRC_ROOT.rglob('*'))
    file_entries = [p for p in entries if p.is_file()]
    total_bytes = sum(p.stat().st_size for p in file_entries)

    print(f'[INFO] Building TAR from {len(file_entries)} files')
    print(f'[INFO] Total input size: {total_bytes / (1024 ** 3):.2f} GB')
    print('[INFO] Source for TAR:', EFFECTIVE_SRC_ROOT)
    t0 = time.time()

    with tarfile.open(LOCAL_TAR_PATH, 'w') as tf:
        if tqdm is None:
            print('[WARN] tqdm is unavailable, fallback without progress bar.')
            for entry in entries:
                arcname = entry.relative_to(EFFECTIVE_SRC_ROOT.parent)
                tf.add(entry, arcname=arcname, recursive=False)
        else:
            with tqdm(total=total_bytes, unit='B', unit_scale=True, desc='TAR build') as pbar:
                for entry in entries:
                    arcname = entry.relative_to(EFFECTIVE_SRC_ROOT.parent)
                    tf.add(entry, arcname=arcname, recursive=False)
                    if entry.is_file():
                        pbar.update(entry.stat().st_size)

    dt = time.time() - t0
    size_mb = LOCAL_TAR_PATH.stat().st_size / (1024 ** 2)
    print(f'[OK] Local TAR created: {LOCAL_TAR_PATH}')
    print(f'[OK] Local TAR size: {size_mb:.1f} MB')
    print(f'[OK] Build time: {dt/60:.1f} min')
else:
    size_mb = LOCAL_TAR_PATH.stat().st_size / (1024 ** 2)
    print(f'[SKIP] Local TAR already exists: {LOCAL_TAR_PATH} ({size_mb:.1f} MB)')

if COPY_TAR_TO_DRIVE:
    DRIVE_TAR_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f'[INFO] Copy TAR to Drive: {LOCAL_TAR_PATH} -> {DRIVE_TAR_PATH}')
    shutil.copy2(LOCAL_TAR_PATH, DRIVE_TAR_PATH)
    drive_size_mb = DRIVE_TAR_PATH.stat().st_size / (1024 ** 2)
    print(f'[OK] Drive TAR ready: {DRIVE_TAR_PATH} ({drive_size_mb:.1f} MB)')


In [ ]:
# Step 4: quick TAR validation (local + optional Drive)
import tarfile

if not LOCAL_TAR_PATH.exists():
    raise FileNotFoundError(f'Local TAR not found: {LOCAL_TAR_PATH}')

with tarfile.open(LOCAL_TAR_PATH, 'r') as tf:
    members = tf.getmembers()

print('members total:', len(members))
print('first 20 members:')
for m in members[:20]:
    print(' -', m.name)

has_train = any('visdrone_yolo/images/train' in m.name for m in members)
has_val = any('visdrone_yolo/images/val' in m.name for m in members)
has_lbl = any('visdrone_yolo/labels/train' in m.name for m in members)
assert has_train and has_val and has_lbl, 'Expected YOLO structure was not found in local TAR.'

print('[OK] Local TAR validation passed')
if COPY_TAR_TO_DRIVE:
    print('[OK] Drive TAR path:', DRIVE_TAR_PATH)


In [ ]:
# Step 5 (optional): test extract TAR to local runtime
RUN_TEST_EXTRACT = False

if RUN_TEST_EXTRACT:
    import shutil
    import subprocess
    from pathlib import Path

    test_dst = Path('/content/datasets_test_extract')
    if test_dst.exists():
        shutil.rmtree(test_dst)
    test_dst.mkdir(parents=True, exist_ok=True)

    cmd = ['tar', '-xf', str(LOCAL_TAR_PATH), '-C', str(test_dst)]
    print('[RUN]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

    print('[OK] Extracted to:', test_dst)
else:
    print('RUN_TEST_EXTRACT=False, skip test extraction.')
